# nb_01_bronze_ingest

**Layer**: Bronze | **Lakehouse**: `lh_bronze_insurance`

**Purpose**: Ingest raw CSV files into Bronze Delta tables with zero transformation. Adds metadata columns (`_source_file`, `_ingested_at`) for lineage tracking.

**Tables Created**:
| Table | Source File | Business Key |
|---|---|---|
| customers_raw | customers.csv | customer_id |
| agents_raw | agents.csv | agent_id |
| policies_raw | policies.csv | policy_id |
| coverages_raw | coverages.csv | coverage_id |
| premiums_raw | premiums.csv | premium_id |
| claims_raw | claims.csv | claim_id |
| claim_payments_raw | claim_payments.csv | payment_id |

**Idempotency**: Uses `overwrite` mode — safe to re-run.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import lit, current_timestamp, input_file_name
from datetime import datetime

# In Fabric, SparkSession is pre-configured. For local, create one.
try:
    spark
except NameError:
    spark = SparkSession.builder \
        .appName("nb_01_bronze_ingest") \
        .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
        .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
        .getOrCreate()

print(f"Spark version: {spark.version}")
print(f"Ingestion started: {datetime.now().isoformat()}")

In [ ]:
# ============================================================
# Configuration
# ============================================================
# In Microsoft Fabric, CSVs should be uploaded to the Files section
# of the lh_bronze_insurance Lakehouse.
# Path resolves to: abfss://<workspace>@onelake.dfs.fabric.microsoft.com/<lakehouse>/Files/seed-data/\n

CSV_BASE_PATH = "Files/seed-data"

# Table definitions: (csv_filename, target_table_name)
TABLE_DEFINITIONS = [
    ("customers.csv",       "customers_raw"),
    ("agents.csv",          "agents_raw"),
    ("policies.csv",        "policies_raw"),
    ("coverages.csv",       "coverages_raw"),
    ("premiums.csv",        "premiums_raw"),
    ("claims.csv",          "claims_raw"),
    ("claim_payments.csv",  "claim_payments_raw"),
]

print(f"Will ingest {len(TABLE_DEFINITIONS)} tables from {CSV_BASE_PATH}")

In [ ]:
# ============================================================
# Ingest all CSVs into Bronze Delta tables
# ============================================================
ingestion_results = []

for csv_file, table_name in TABLE_DEFINITIONS:
    csv_path = f"{CSV_BASE_PATH}/{csv_file}"
    print(f"\n{'='*60}")
    print(f"Ingesting: {csv_file} → {table_name}")
    print(f"{'='*60}")
    
    try:
        # Read CSV with header, infer schema (all strings in Bronze)
        df = spark.read.format("csv") \
            .option("header", "true") \
            .option("inferSchema", "false") \
            .load(csv_path)
        
        # Add metadata columns for lineage
        df = df.withColumn("_source_file", lit(csv_file)) \
               .withColumn("_ingested_at", current_timestamp())
        
        row_count = df.count()
        col_count = len(df.columns)
        
        # Write as Delta table (overwrite for idempotency)
        df.write.format("delta") \
            .mode("overwrite") \
            .option("overwriteSchema", "true") \
            .saveAsTable(table_name)
        
        ingestion_results.append((table_name, row_count, col_count, "SUCCESS"))
        print(f"  ✓ {row_count:,} rows × {col_count} columns written to {table_name}")
        
    except Exception as e:
        ingestion_results.append((table_name, 0, 0, f"FAILED: {str(e)}"))
        print(f"  ✗ FAILED: {str(e)}")

print(f"\nIngestion complete: {sum(1 for r in ingestion_results if r[3] == 'SUCCESS')}/{len(TABLE_DEFINITIONS)} tables succeeded")

In [ ]:
# ============================================================
# Validation: Read back and verify row counts
# ============================================================
print("BRONZE INGESTION VALIDATION")
print("=" * 60)
print(f"{'Table':<25} {'Rows':>8} {'Cols':>6} {'Status':>10}")
print("-" * 60)

for table_name, row_count, col_count, status in ingestion_results:
    print(f"{table_name:<25} {row_count:>8,} {col_count:>6} {status:>10}")

total_rows = sum(r[1] for r in ingestion_results)
print("-" * 60)
print(f"{'TOTAL':<25} {total_rows:>8,}")
print()

# Show sample from each table
for table_name, _, _, status in ingestion_results:
    if status == "SUCCESS":
        print(f"\n--- Sample: {table_name} ---")
        spark.table(table_name).show(3, truncate=30)